In [1]:
!pip install transformers
!pip install git+https://github.com/openai/CLIP.git

  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-tnlx78d_
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-tnlx78d_
  Resolved https://github.com/openai/CLIP.git to commit dcba3cb2e2827b402d2701e7e1c7d9fed8a20ef1
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.9 MB/s eta 0:00:00
  Created wheel for clip: filename=clip-1.0-py3-none-any.whl size=1369490 sha256=869e56a2fca89b5c837b5c3fb9b3bd4138a4a6ca0922fa28f6e6307d9daef09f
  Stored in directory: /tmp/pip-ephem-wheel-cache-iwnvteei/wheels/35/3e/df/3d24cbfb3b6a06f17a2bfd7d1138900d4365d9028aa8f6e92f
Successfully built clip


In [1]:
!unzip {'/content/sample_data/vi.zip'} -d {'/content/sample_data/image'}

Archive:  /content/sample_data/vi.zip
   creating: /content/sample_data/image/vi/
  inflating: /content/sample_data/image/vi/1.jpg  
  inflating: /content/sample_data/image/vi/10.jpg  
  inflating: /content/sample_data/image/vi/11.jpg  
  inflating: /content/sample_data/image/vi/12.jpg  
  inflating: /content/sample_data/image/vi/13.jpg  
  inflating: /content/sample_data/image/vi/14.jpg  
  inflating: /content/sample_data/image/vi/15.jpg  
  inflating: /content/sample_data/image/vi/16.jpg  
  inflating: /content/sample_data/image/vi/17.jpg  
  inflating: /content/sample_data/image/vi/18.jpg  
  inflating: /content/sample_data/image/vi/19.jpg  
  inflating: /content/sample_data/image/vi/2.jpg  
  inflating: /content/sample_data/image/vi/20.jpg  
  inflating: /content/sample_data/image/vi/21.jpg  
  inflating: /content/sample_data/image/vi/22.jpg  
  inflating: /content/sample_data/image/vi/23.jpg  
  inflating: /content/sample_data/image/vi/24.jpg  
  inflating: /content/sample_data/im

In [2]:
import torch
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration
import os

dataset_path = "/content/sample_data/image/vi"
filenames = os.listdir(dataset_path)
filenames.sort()
print(len(filenames))

# Define file_path once outside the loop
file_path = "/content/sample_data/image/vi_text"

# Check if file_path exists as a file and remove it
if os.path.isfile(file_path):
    os.remove(file_path)

# Create the directory if it doesn't exist
os.makedirs(file_path, exist_ok=True)

for file in filenames:  # 遍历二级文件名称，读取二级文件里面的内容
    img_path = os.path.join(dataset_path, file)  # 获取二级路径下的文件名称
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-large")
    model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-large").to(device)
    raw_image = Image.open(img_path).convert('RGB')
    inputs = processor(raw_image, return_tensors="pt").to(device)
    out = model.generate(**inputs)

    text = processor.decode(out[0], skip_special_tokens=True)

    text_path = os.path.join(file_path, file.split(".")[0] + ".txt")

    print(file.split(".")[0] + ": "+ text)
    img_name = file

    # 写入单个文本文件
    with open(text_path, 'w') as f_individual:
        f_individual.write(text)

    # 写入汇总文本文件 (now in the vi_text directory)
    with open(os.path.join(file_path, "vi.txt"), 'a') as f_summary:
        f_summary.write(str(img_name.split(".")[0]) + "&&" + str(text) + "\n" )

print("done")


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


80


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/445 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/527 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.88G [00:00<?, ?B/s]

1: there are two people walking in the parking lot at night
10: there are two people walking down the street near a bench
11: there are many bikes parked on the side of the street
12: there are many blue cones on the street near a building
13: there is a man walking across the street with a suitcase
14: there is a red and white ambulance driving down the street
15: there are two trucks that are driving down the street
16: there are people walking down the street with a dog on a leash
17: there is a street with a yellow line on the side of it
18: people walking down a street with cars and buildings in the background
19: there is a woman walking down the street with a red umbrella
2: cars parked in a parking lot in front of a building
20: cars parked in a parking lot in front of a building
21: there is a man walking down the street with a cell phone
22: there are people walking down the street in the city
23: there is a man riding a motorcycle down the street
24: there is a person walkin

In [3]:
import zipfile


def zip_ya(startdir, file_news):
    z = zipfile.ZipFile(file_news, 'w', zipfile.ZIP_DEFLATED)  # 参数一：文件夹名
    for dirpath, dirnames, filenames in os.walk(startdir):
        fpath = dirpath.replace(startdir, '')
        fpath = fpath and fpath + os.sep or ''
        for filename in filenames:
            z.write(os.path.join(dirpath, filename), fpath + filename)
    print('压缩成功')
    z.close()


if __name__ == "__main__":
    startdir = "/content/sample_data/image/vi_text"  # 要压缩的文件夹路径
    file_news = startdir + '.zip'  # 压缩后文件夹的名字
    zip_ya(startdir, file_news)

压缩成功
